# Cross-Species Neural Alignment

Aligning neural representations across species to understand evolutionary conservation and specialization.

**Methods covered:**
- Procrustes alignment
- Representational similarity analysis (RSA)
- Phylogenetic distance correlation
- Conserved vs species-specific decomposition

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from neuros_mechint.alignment import (
    ProcrustesAlignment,
    CrossSpeciesRSA,
    PhylogeneticDistance,
    ConservedSpecificDecomposition
)
from neuros_mechint.visualization import CrossSpeciesVisualizer

%matplotlib inline

## 1. Procrustes Alignment

Align two neural spaces using optimal rotation, scaling, and translation.

In [ ]:
# Simulate neural representations from two species
# (In practice, these would be activations from models trained on different species)
n_stimuli = 50
n_dims = 10

# Mouse visual cortex (V1)
mouse_V1 = np.random.randn(n_stimuli, n_dims)

# Macaque visual cortex (V1) - similar but with transformation
# Add rotation + noise to simulate evolutionary divergence
angle = np.pi / 6  # 30 degrees
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
macaque_V1 = mouse_V1.copy()
macaque_V1[:, :2] = macaque_V1[:, :2] @ R.T
macaque_V1 += np.random.randn(n_stimuli, n_dims) * 0.5

# Perform Procrustes alignment
aligner = ProcrustesAlignment()
aligned_mouse, disparity = aligner.align(mouse_V1, macaque_V1)

print(f"Procrustes disparity: {disparity:.4f}")
print(f"Scale factor: {aligner.scale:.4f}")
print(f"Rotation matrix shape: {aligner.R.shape}")

# Visualize
viz = CrossSpeciesVisualizer()
fig = viz.plot_procrustes_alignment(
    source=mouse_V1,
    target=macaque_V1,
    aligned=aligned_mouse,
    species_names=("Mouse V1", "Macaque V1")
)
fig.show()

## 2. Representational Similarity Analysis (RSA)

Compare representational geometries using RDMs.

In [ ]:
# Create multiple species representations
species_reps = {
    'Mouse': np.random.randn(n_stimuli, n_dims),
    'Rat': np.random.randn(n_stimuli, n_dims),
    'Macaque': np.random.randn(n_stimuli, n_dims),
    'Human': np.random.randn(n_stimuli, n_dims)
}

# Add some shared structure (simulate conserved representations)
shared_signal = np.random.randn(n_stimuli, n_dims) * 2
for name in species_reps:
    species_reps[name] += shared_signal * 0.3

# Compute RSA
rsa = CrossSpeciesRSA()
similarity_matrix = rsa.compute_similarity_matrix(species_reps)

# Visualize RDM for one species
from scipy.spatial.distance import pdist, squareform
mouse_rdm = squareform(pdist(species_reps['Mouse'], metric='correlation'))

viz = CrossSpeciesVisualizer()
fig = viz.plot_rsa_matrix(
    rdm=mouse_rdm,
    labels=[f"Stim{i}" for i in range(n_stimuli)] if n_stimuli <= 20 else None,
    title="Mouse V1 Representational Dissimilarity Matrix"
)
fig.show()

# Cross-species similarity
species_names = list(species_reps.keys())
fig2 = viz.plot_phylogenetic_comparison(
    species_names=species_names,
    similarity_matrix=similarity_matrix,
    title="Cross-Species Neural Similarity"
)
fig2.show()

print("\nCross-Species Similarity Matrix:")
print("     ", " ".join([f"{s:8s}" for s in species_names]))
for i, s1 in enumerate(species_names):
    print(f"{s1:8s}", " ".join([f"{similarity_matrix[i,j]:.4f}  " for j in range(len(species_names))]))

## 3. Phylogenetic Distance Correlation

Test if neural similarity decreases with evolutionary distance.

In [ ]:
# Define phylogenetic tree (simplified)
phylo = PhylogeneticDistance()

# Compute pairwise phylogenetic distances (in millions of years)
phylo_dist_matrix = np.array([
    [0,   10,  85, 90],  # Mouse
    [10,  0,   85, 90],  # Rat
    [85,  85,  0,  25],  # Macaque
    [90,  90,  25, 0 ]   # Human
])

# Plot correlation
fig = viz.plot_phylogenetic_comparison(
    species_names=species_names,
    similarity_matrix=similarity_matrix,
    phylo_distances=phylo_dist_matrix,
    title="Neural Similarity vs Phylogenetic Distance"
)
fig.show()

# Statistical test
from scipy.stats import spearmanr

# Extract upper triangle
triu_indices = np.triu_indices(len(species_names), k=1)
neural_sim = similarity_matrix[triu_indices]
phylo_dist = phylo_dist_matrix[triu_indices]

rho, pval = spearmanr(-phylo_dist, neural_sim)  # Negative because greater distance = less similarity

print(f"\nPhylogenetic correlation:")
print(f"Spearman ρ = {rho:.3f}")
print(f"p-value = {pval:.4f}")
print(f"\nInterpretation: {'Significant' if pval < 0.05 else 'Not significant'} correlation")
print(f"between phylogenetic distance and neural similarity.")

## 4. Conserved vs Species-Specific Decomposition

Decompose representations into shared and unique components.

In [ ]:
# Use CSD to find conserved and specific components
csd = ConservedSpecificDecomposition(n_conserved=5, n_specific=3)

# Fit on all species
species_list = [species_reps[name] for name in species_names]
result = csd.decompose(species_list)

# Visualize variance explained
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
conserved_var = result['conserved_variance']
specific_vars = result['specific_variance']

x = np.arange(len(species_names))
plt.bar(x, conserved_var, label='Conserved', alpha=0.7)
bottom = conserved_var
for i, name in enumerate(species_names):
    plt.bar(x[i], specific_vars[i], bottom=bottom[i], 
            label=f'{name}-specific' if i == 0 else '', alpha=0.7)

plt.xticks(x, species_names)
plt.ylabel('Variance Explained')
plt.title('Conserved vs Species-Specific Variance')
plt.legend()
plt.grid(True, alpha=0.3)

# Conserved dimensions
plt.subplot(1, 2, 2)
conserved_dims = result['conserved_components']
plt.imshow(conserved_dims[:, :5].T, aspect='auto', cmap='RdBu_r')
plt.colorbar(label='Component Weight')
plt.xlabel('Original Dimension')
plt.ylabel('Conserved Component')
plt.title('Conserved Components (Top 5)')

plt.tight_layout()
plt.show()

print("\nDecomposition Results:")
for i, name in enumerate(species_names):
    total_var = conserved_var[i] + specific_vars[i]
    print(f"{name}:")
    print(f"  Conserved: {conserved_var[i]/total_var*100:.1f}%")
    print(f"  Specific:  {specific_vars[i]/total_var*100:.1f}%")

## Conclusion

This notebook demonstrated:
1. **Procrustes alignment**: Optimal transformation between species
2. **RSA**: Comparing representational geometries
3. **Phylogenetic correlation**: Linking evolution to neural similarity
4. **CSD**: Decomposing conserved and specific components

These methods enable rigorous cross-species comparisons to identify evolutionarily conserved neural computations.